In [1]:
import os

dirs = [
    "app",
    "app/db",
    "app/models",
    "app/services",
    "app/routes",
    "ingestion",
    "ingestion/loaders",
    "data",
    "data/raw",
    "data/processed",
    "notebooks"
]

for d in dirs:
    os.makedirs(d, exist_ok=True)


In [2]:
files = [
    "app/__init__.py",
    "app/db/__init__.py",
    "app/models/__init__.py",
    "app/services/__init__.py",
    "app/routes/__init__.py",
    "ingestion/__init__.py",
    "ingestion/loaders/__init__.py"
]

for f in files:
    open(f, "w").close()


In [3]:
%%writefile app/db/connection.py
import psycopg2
from psycopg2.extras import RealDictCursor

def get_connection():
    return psycopg2.connect(
        dbname="digimon",
        user="postgres",
        password="CHANGE_ME",
        host="localhost",
        port=5432,
        cursor_factory=RealDictCursor
    )


Writing app/db/connection.py


In [4]:
%%writefile ingestion/cards.py
import requests
from psycopg2.extras import execute_values
from app.db.connection import get_connection

API_URL = "https://digimoncard.io/api-public/search.php"

def fetch_all_cards():
    print("Fetching cards from DigimonCard.io...")

    response = requests.get(API_URL, params={"sort": "cardnumber"})
    response.raise_for_status()

    cards = response.json()
    print(f"Fetched {len(cards)} cards.")
    return cards

def ingest_cards(cards):
    conn = get_connection()
    cur = conn.cursor()

    rows = []
    for c in cards:
        rows.append((
            c["cardnumber"],
            c["name"],
            c["color"].split("/") if c.get("color") else [],
            int(c["level"]) if c.get("level") else None,
            c.get("type"),
            c["form"].split("/") if c.get("form") else [],
            c.get("effect"),
            c.get("set_name"),
            True,
            "legal",
            []
        ))

    execute_values(cur, """
        INSERT INTO cards
        (card_id, name, color, level, type, traits, effect_text, set_name, is_legal, ban_status, tags)
        VALUES %s
        ON CONFLICT (card_id) DO NOTHING
    """, rows)

    conn.commit()
    cur.close()
    conn.close()

    print("Card ingestion complete.")


Writing ingestion/cards.py


In [6]:
%%writefile ingestion/loaders/ingestion.py
from ingestion.cards import fetch_all_cards, ingest_cards

def run_card_ingestion():
    cards = fetch_all_cards()
    ingest_cards(cards)
    print("Done.")


Overwriting ingestion/loaders/ingestion.py


In [ ]:
from ingestion.loaders.ingestion import run_card_ingestion
run_card_ingestion()
